# Build EHR Fusion Dataset — ragged category-ids + masked continuous

Redesigns the **EHR features** of the fusion dataset into the data contract of
**PLAN_cxr_ehr_film_fusion.md** (§2, §5.2) and writes a **new** dataset.
Everything is in this notebook — no `fusion/` package.

**EHR provenance:** MIMIC-IV v3.1 on BigQuery (project `mimic-ehr-v310`, datasets
`physionet-data.mimiciv_3_1_hosp` / `_icu`), pulled by `EHR_retrieval.ipynb`.
Organism / specimen come from `microbiologyevents` (`org_name`,
`spec_type_desc`); labs from `labevents`; vitals from ICU chartevents.

**Input** — `fusion_dataset_hybrid_fullmicro.csv` (the **full-microbiology**
build): `img_0..img_767` (frozen RAD-DINO CLS) · `org_*` (**617**) / `spec_*`
(**95**) multi-hot · flat `lab_*`/`vital_*`/`age` · binary flags · `y_*` (14
CheXpert). We take **all** org/spec now — no one-hot dimensionality cost with
`EmbeddingBag`; the pruned 20/15 `..._full.csv` is no longer used.

**Output** (new files, nothing overwritten in place):

| file | contents |
|---|---|
| `cxr_emb.npy` | `[N,768]` float32 — frozen CXR embedding, row-aligned |
| `ehr_arrays.npz` | `labs[N,L]`, `labs_mask[N,L]`, `y[N,14]` |
| `ehr_meta.parquet` | `dicom_id, subject_id, study_id, split, setting, view` + **ragged** `org_ids`, `spec_ids` |
| `ehr_config.pkl` | vocabs, column order, standardizer stats, label names |

Vocabs/standardizer are fit on **train only**. Reserved ids `PAD=0, UNK=1, MISSING=2`.


## C0 · Setup & config

Mount Drive on Colab; set paths and the feature contract in one place.

In [ ]:
try:
    import pyarrow            # noqa: F401  (parquet engine; preinstalled on Colab)
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','pyarrow'])

import os, sys, glob, pickle, json, numpy as np, pandas as pd
from collections import Counter

# --- Colab Drive mount. NOTE: do NOT swallow mount errors into a silent
# local fallback -- that is how you end up reading a stale './...csv'. ---
IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')            # let it raise if it fails
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'   # <-- your data folder
else:
    DATA_DIR = '.'
print('DATA_DIR =', DATA_DIR)

# >>> SET THIS to your CURRENT fullmicro CSV (617 org_ + 95 spec_). Run the next
# >>> cell first if you don't know the path -- it lists candidates on Drive.
CSV_PATH = os.path.join(DATA_DIR, 'fusion_dataset_hybrid_fullmicro.csv')
OUT_DIR  = os.path.join(DATA_DIR, 'ehr_fusion_v1')   # NEW folder, nothing overwritten
os.makedirs(OUT_DIR, exist_ok=True)

QUICK = False          # True -> only first QUICK_N rows, for a fast dry run
QUICK_N = 20_000

# reserved categorical ids (PLAN §2)
PAD, UNK, MISSING = 0, 1, 2

# NOTE: these are WANTED lists. C1 resolves them against the real header and
# drops anything absent, so a newer build with extra/renamed columns still runs.
# Anything in the file we don't claim is reported by C1 for you to triage.
LAB_COLS  = ['lab_hemoglobin','lab_wbc','lab_rbc','lab_platelet','lab_creatinine',
             'lab_sodium','lab_potassium','lab_glucose','lab_bun','lab_chloride']
VITAL_COLS= ['vital_heart_rate','vital_spo2','vital_temperature','vital_resprate',
             'vital_sbp','vital_dbp']
DENSE_COLS= ['age','n_positive_cultures']
CONT_WANTED = LAB_COLS + VITAL_COLS + DENSE_COLS    # order is frozen in config

# right-skewed -> log1p (WBC high = leukemia signal, NOT an outlier: memory note).
LOG_COLS  = {'lab_wbc','lab_platelet','lab_glucose','lab_creatinine','lab_bun',
             'n_positive_cultures'}
WINSOR_PCT = (0.5, 99.5)     # applied to non-log continuous only

# always-present binary flags -> appended to the continuous block with mask=1.
# Newer builds add comorbidities (e.g. HIV) -- listed here and existence-filtered
# in C1. Add any name C1 reports as unclaimed that you want as a feature.
FLAG_SPEC = [('gender','F')]            # gender==F -> 1
FLAG_BIN_WANTED = ['ed_cough','smoke','cough','diabetes',
                   'hiv','dx_hiv','has_hiv']        # HIV name varies by build

LABELS = ['Atelectasis','Cardiomegaly','Consolidation','Edema',
          'Enlarged Cardiomediastinum','Fracture','Lung Lesion','Lung Opacity',
          'No Finding','Pleural Effusion','Pleural Other','Pneumonia',
          'Pneumothorax','Support Devices']
Y_COLS = ['y_'+l for l in LABELS]

print('CSV :', CSV_PATH)
print('OUT :', OUT_DIR)
print('wanted continuous:', len(CONT_WANTED), '| wanted flags:',
      len(FLAG_SPEC)+len(FLAG_BIN_WANTED), '(resolved against the header in C1)')

## C0-bis · Find the dataset on Drive (run if `CSV_PATH` is wrong)

Lists candidate fusion datasets / embedding caches on Drive with size + mtime so
you can point `CSV_PATH` at the **newest** build. This notebook never guesses a
path silently — if `CSV_PATH` doesn't exist, the next cell stops with an error
instead of falling back to a stale local file.

In [ ]:
roots = [DATA_DIR] + sorted(glob.glob(os.path.join(DATA_DIR,'*')))[:200]
pats  = ['fusion_dataset*.csv','*fusion*.csv','*emb*.npy','*.parquet']
seen, hits = set(), []
for r in roots:
    if not os.path.isdir(r): continue
    for p in pats:
        for f in glob.glob(os.path.join(r,p)):
            if f in seen: continue
            seen.add(f); hits.append(f)
import datetime as _dt
if hits:
    print(f'{"size(MB)":>10}  {"modified":<17} path')
    for f in sorted(hits, key=lambda x:-os.path.getmtime(x)):
        sz=os.path.getsize(f)/1e6
        mt=_dt.datetime.fromtimestamp(os.path.getmtime(f)).strftime('%Y-%m-%d %H:%M')
        print(f'{sz:10.1f}  {mt:<17} {f}')
else:
    print('no candidates found under', DATA_DIR, '- check the Drive folder')
print('\nCurrent CSV_PATH =', CSV_PATH, '| exists:', os.path.exists(CSV_PATH))

## C1 · Load the non-image columns — **every** `org_`/`spec_` included

Reads all columns except the 768 `img_*` (those stream straight to `.npy` in C7).
For the fullmicro build that is ~769 columns: **all 617 `org_` + all 95 `spec_`**
plus labs/vitals/flags/labels. Nothing is filtered out here — `META_COLS` is
literally "everything that isn't an image column", so no organism is dropped.
It stays cheap because the wide part (the embedding) is excluded, which lets
vocab-building and standardization run fully in memory.

In [ ]:
# fail loudly rather than silently reading some other/stale file
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f'CSV_PATH does not exist: {CSV_PATH}\n'
        'Run C0-bis to list candidates on Drive, then set CSV_PATH in C0.')

head = pd.read_csv(CSV_PATH, nrows=0)
ALL_COLS = list(head.columns)
IMG_COLS = [c for c in ALL_COLS if c.startswith('img_')]
ORG_COLS = [c for c in ALL_COLS if c.startswith('org_')]
SPEC_COLS= [c for c in ALL_COLS if c.startswith('spec_')]
assert len(IMG_COLS) == 768, f'expected 768 img_ cols, got {len(IMG_COLS)}'
print('img', len(IMG_COLS), '| org', len(ORG_COLS), '| spec', len(SPEC_COLS))
# the pruned build has 20 org_/15 spec_; the fullmicro build has ~617/~95
if len(ORG_COLS) < 100:
    print(f'\n!! WARNING: only {len(ORG_COLS)} org_ columns -- this looks like the '
          'PRUNED one-hot build, not fullmicro. Point CSV_PATH at the '
          'full-microbiology CSV to get all organisms.')

META_COLS = [c for c in ALL_COLS if not c.startswith('img_')]
nrows = QUICK_N if QUICK else None
meta = pd.read_csv(CSV_PATH, usecols=META_COLS, nrows=nrows, low_memory=False)
N = len(meta)
other = len(META_COLS) - len(ORG_COLS) - len(SPEC_COLS)
print(f'rows loaded: {N}')
print(f'non-image cols loaded: {len(META_COLS)} = {len(ORG_COLS)} org_ + '
      f'{len(SPEC_COLS)} spec_ + {other} other  <- ALL org/spec, none dropped')
assert len(ORG_COLS)+len(SPEC_COLS)+other == len(META_COLS)

# ---- resolve WANTED feature lists against the real header ----------------
CONT_COLS = [c for c in CONT_WANTED       if c in ALL_COLS]
FLAG_BIN  = [c for c in FLAG_BIN_WANTED   if c in ALL_COLS]
if set(CONT_WANTED)-set(CONT_COLS):
    print('!! continuous cols NOT in this file (skipped):', sorted(set(CONT_WANTED)-set(CONT_COLS)))
print('flags used:', FLAG_BIN)

# ---- report every column we do NOT use, so nothing is silently dropped ----
claimed = set(CONT_COLS)|set(FLAG_BIN)|set(Y_COLS)|{c for c,_ in FLAG_SPEC}|set(ORG_COLS)|set(SPEC_COLS)|{
    'dicom_id','subject_id','study_id','split','view','hadm_id','stay_id',
    'study_dt','admittime','ed_intime','encounter_time'}
unclaimed = [c for c in META_COLS if c not in claimed]
if unclaimed:
    print('\n>> UNCLAIMED columns (present in the CSV but NOT used as features):')
    for c in unclaimed:
        u = meta[c].dropna().unique()
        kind = 'binary' if set(u.tolist()) <= {0,1,0.0,1.0} and len(u)<=2 else f'{len(u)} vals'
        print(f'   {c:<28} {kind}')
    print('   -> add any of these to FLAG_BIN_WANTED / CONT_WANTED in C0 to use them.')

# normalise split labels -> train/val/test
meta['split'] = meta['split'].replace({'validate':'val','valid':'val','dev':'val'})
print(meta['split'].value_counts().to_dict())
train_mask = (meta['split'] == 'train').values
print('train rows:', int(train_mask.sum()))

## C2 · Subject-level leakage check + `setting` (ED vs inpatient)

Assert `subject_id` sets are disjoint across splits (PLAN §7.2). Derive a
`setting` column for the inpatient/ED stratified Δ in PLAN §6.

In [ ]:
sets = {s: set(meta.loc[meta.split==s,'subject_id']) for s in ['train','val','test']}
for a in sets:
    for b in sets:
        if a<b:
            inter = sets[a] & sets[b]
            print(f'{a}∩{b}: {len(inter)} shared subjects')
            assert len(inter)==0, f'LEAKAGE between {a} and {b}'
print('no subject-level leakage ✔')

# ED encounter carries a stay_id; pure inpatient does not
meta['setting'] = np.where(meta.get('stay_id', pd.Series(np.nan,index=meta.index)).notna(),
                           'ED','inpatient')
print(meta['setting'].value_counts().to_dict())

## C2-bis · AUDIT — are `MISSING` / `NO_GROWTH` / `CANCELLED` real, separable states?

**Read-only. Changes nothing.** The current pipeline collapses "no positive
organism" into a single `MISSING` id. That may be wrong: clinically,

| proposed | meaning | how we'd detect it in this CSV |
|---|---|---|
| `MISSING` | no microbiology at all — never suspected | no `spec_*` **and** no `org_*` |
| `NO_GROWTH` | specimen sent, nothing grew → **negative** | some `spec_*`, no real `org_*` |
| `CANCELLED` | specimen sent, culture cancelled | `org_CANCELLED` set |

"Never tested" and "tested, came back clean" are *opposite* signals — a doctor
ordering a culture is itself informative (the MNAR argument PLAN §5.2 makes for
labs). This cell tests whether the three states are (a) actually present,
(b) separable, and (c) internally consistent with `n_positive_cultures`.

In [ ]:
import re
otok = {c: c[len('org_'):]  for c in ORG_COLS}

# ---- 1. which org_ "organisms" are actually STATUS / non-specific strings? ----
STATUS_RE = re.compile(r'^(CANCELLED|POSITIVE|NEGATIVE|.*ISOLATE.*|'
                       r'.*GROWN_IN_CULTURE.*|.*INFORMATION.*|.*FOLLOW.*)$')
VAGUE_RE  = re.compile(r'^(MIXED_.*|.*GRAM_(NEGATIVE|POSITIVE)_ROD.*|.*GRAM_.*COCC.*|'
                       r'.*PRESUMPTIVE.*|MOLD|YEAST)$')
status_cols = [c for c in ORG_COLS if STATUS_RE.match(otok[c])]
vague_cols  = [c for c in ORG_COLS if VAGUE_RE.match(otok[c]) and c not in status_cols]

def freq(c): return int(meta[c].fillna(0).astype(bool).sum())
print('=== org_ tokens that are a STATUS, not an organism ===')
for c in sorted(status_cols, key=lambda c:-freq(c)):
    print(f'   {otok[c]:<58} {freq(c):>7} rows ({freq(c)/N:.1%})')
print('\n=== org_ tokens that are NON-SPECIFIC descriptors (judgement call) ===')
for c in sorted(vague_cols, key=lambda c:-freq(c))[:12]:
    print(f'   {otok[c]:<58} {freq(c):>7} rows ({freq(c)/N:.1%})')

real_org_cols = [c for c in ORG_COLS if c not in status_cols]
print(f'\norg_ columns: {len(ORG_COLS)} total = {len(real_org_cols)} real + {len(status_cols)} status')

In [ ]:
# ---- 2. are the three states separable? ----
any_real = meta[real_org_cols].fillna(0).astype(bool).any(axis=1)
any_spec = meta[SPEC_COLS].fillna(0).astype(bool).any(axis=1) if SPEC_COLS else pd.Series(False,index=meta.index)
cancel_c = [c for c in ORG_COLS if otok[c]=='CANCELLED']
cancelled= meta[cancel_c].fillna(0).astype(bool).any(axis=1) if cancel_c else pd.Series(False,index=meta.index)

print('=== cross-tab: specimen collected  x  real organism grown ===')
print(pd.crosstab(any_spec.rename('any_spec_'), any_real.rename('any_real_org')))

state = np.select(
    [any_real, cancelled & ~any_real, any_spec & ~any_real & ~cancelled],
    ['POSITIVE', 'CANCELLED', 'NO_GROWTH'], default='MISSING')
state = pd.Series(state, index=meta.index)
print('\n=== proposed 4-way state ===')
vc = state.value_counts()
for k,v in vc.items(): print(f'   {k:<10} {v:>8} ({v/N:.1%})')
print(f'\n current pipeline lumps NO_GROWTH+CANCELLED+MISSING into ONE id (MISSING) = '
      f'{(state!="POSITIVE").sum()/N:.1%} of studies')

In [ ]:
# ---- 3. consistency: does n_positive_cultures agree with the state? ----
if 'n_positive_cultures' in meta:
    npc = meta['n_positive_cultures'].fillna(0).astype(float)
    print('=== n_positive_cultures by state (expect 0 for NO_GROWTH/MISSING) ===')
    print(pd.DataFrame({'mean':npc.groupby(state).mean().round(2),
                        'median':npc.groupby(state).median(),
                        'frac_gt0':npc.gt(0).groupby(state).mean().round(3),
                        'n':npc.groupby(state).size()}))
    bad = ((state.isin(['NO_GROWTH','MISSING'])) & (npc>0)).sum()
    print(f'\n!! CONTRADICTION: {bad} rows ({bad/N:.1%}) have no organism but '
          f'n_positive_cultures>0' if bad else '\n consistent: no-organism rows all have n_positive_cultures==0 ✔')

# ---- 4. anomalies ----
anom = (any_real & ~any_spec).sum()
print(f'\norganism grown but NO specimen recorded: {anom} rows ({anom/N:.1%})'
      + ('  <- impossible, join is suspect' if anom else '  ✔'))
if cancel_c:
    both=(cancelled & any_real).sum()
    print(f'CANCELLED co-occurring with a real organism: {both} rows ({both/N:.1%})'
          '  -> CANCELLED is per-specimen, not per-study' if both else
          f'CANCELLED never co-occurs with a real organism ✔')

## C3 · Categorical vocabs (train-only) → ragged ids

`org_*` / `spec_*` are fixed multi-hot columns. Build a vocab over the columns
that are **positive at least once in train**; a column with zero train positives
is treated as `UNK` at encode time. A study with no positive → single `MISSING`
id. This is the "keep the feature, drop the explicit one-hot" step (PLAN §5.2).

In [ ]:
def build_vocab(cols, prefix):
    # train positive count per column, frequency-ranked; reserved ids first
    cnt = {c: int(meta.loc[train_mask, c].fillna(0).astype(bool).sum()) for c in cols}
    ranked = [c for c,_ in sorted(cnt.items(), key=lambda kv:-kv[1]) if cnt[c] > 0]
    col2id = {c: i+3 for i,c in enumerate(ranked)}          # ids start after reserved
    zero_train = [c for c in cols if cnt[c]==0]             # -> UNK at encode
    id2name = {PAD:'<PAD>', UNK:'<UNK>', MISSING:'<MISSING>'}
    id2name.update({v:k[len(prefix):] for k,v in col2id.items()})
    print(f'{prefix} vocab: {len(col2id)} real ids (+3 reserved); '
          f'{len(zero_train)} cols zero-in-train -> UNK')
    return {'col2id':col2id, 'zero_train':set(zero_train), 'id2name':id2name}

org_vocab  = build_vocab(ORG_COLS,  'org_')
spec_vocab = build_vocab(SPEC_COLS, 'spec_')

def encode_ragged(cols, vocab):
    """multi-hot rows -> list[list[int]]; empty -> [MISSING]; unknown -> UNK."""
    sub = meta[cols].fillna(0).astype(bool).values
    col2id = vocab['col2id']
    ids_per_col = [col2id[c] if c in col2id else UNK for c in cols]  # zero-train -> UNK
    out = []
    for row in sub:
        ids = sorted({ids_per_col[j] for j in np.nonzero(row)[0]})
        out.append(ids if ids else [MISSING])
    return out

org_ids  = encode_ragged(ORG_COLS,  org_vocab)
spec_ids = encode_ragged(SPEC_COLS, spec_vocab)

# sanity: a multi-value row and a MISSING row
mv = max(range(N), key=lambda i: len(org_ids[i]))
print('max-org study ids:', org_ids[mv], '->',
      [org_vocab['id2name'][i] for i in org_ids[mv]])
print('frac studies with MISSING org:', np.mean([o==[MISSING] for o in org_ids]).round(3))

## C4 · Continuous labs/vitals → standardized + mask

Per PLAN §5.2: train stats only, log1p on skewed labs, winsorize the rest,
median-impute missing, and emit `labs_mask` (the MNAR "was it ordered" signal).

In [ ]:
def fit_standardizer(df, train_mask):
    stats = {}
    for c in CONT_COLS:
        raw = df[c].astype(float).values
        obs = raw[train_mask & ~np.isnan(raw)]
        x = np.log1p(np.clip(obs,0,None)) if c in LOG_COLS else obs.copy()
        if c in LOG_COLS:
            lo, hi = -np.inf, np.inf                       # log already compresses tail
        else:
            lo, hi = np.percentile(x, WINSOR_PCT)          # winsor bounds from train
            x = np.clip(x, lo, hi)
        med = float(np.median(x)); mu = float(x.mean()); sd = float(x.std() or 1.0)
        stats[c] = dict(log=c in LOG_COLS, lo=float(lo), hi=float(hi),
                        median=med, mean=mu, std=sd)
    return stats

def apply_standardizer(df, stats):
    L = len(CONT_COLS)
    labs = np.zeros((len(df), L), np.float32)
    mask = np.zeros((len(df), L), np.float32)
    for j,c in enumerate(CONT_COLS):
        raw = df[c].astype(float).values
        obs = ~np.isnan(raw)
        s = stats[c]
        x = np.where(obs, raw, np.nan)
        if s['log']: x = np.log1p(np.clip(x,0,None))
        x = np.clip(x, s['lo'], s['hi'])
        x = np.where(obs, x, s['median'])                  # median-impute
        labs[:,j] = ((x - s['mean'])/s['std']).astype(np.float32)
        mask[:,j] = obs.astype(np.float32)
    return labs, mask

cont_stats = fit_standardizer(meta, train_mask)
cont, cont_mask = apply_standardizer(meta, cont_stats)
print('cont', cont.shape, 'mean~0 std~1 on train:',
      cont[train_mask].mean().round(3), cont[train_mask].std().round(3))
print('missing-rate per col (mask=0):',
      dict(zip(CONT_COLS, (1-cont_mask.mean(0)).round(2))))

## C5 · Binary flags → appended to the continuous block

`gender`, `ed_cough`, `smoke`, `cough`, `diabetes` are complete binaries. They
join the continuous block as extra dims with `mask=1`, giving the final
`labs [N,L]` / `labs_mask [N,L]` of the data contract.

In [ ]:
flag_cols_out = [f'{c}_{v}' for c,v in FLAG_SPEC] + FLAG_BIN
flags = np.zeros((N, len(flag_cols_out)), np.float32)
k = 0
for c,v in FLAG_SPEC:
    flags[:,k] = (meta[c].astype(str).values == v).astype(np.float32); k+=1
for c in FLAG_BIN:
    flags[:,k] = meta[c].fillna(0).astype(float).clip(0,1).values.astype(np.float32); k+=1

labs      = np.concatenate([cont, flags], axis=1).astype(np.float32)
labs_mask = np.concatenate([cont_mask, np.ones_like(flags)], axis=1).astype(np.float32)
LABS_COLS = CONT_COLS + flag_cols_out
L = labs.shape[1]
print('final labs', labs.shape, '| L =', L)
print('cols:', LABS_COLS)

## C6 · Labels `y [N,14]`

Values are already binarized 0/1 upstream (verified — no `-1` uncertains), so no
U-policy step is needed; `NaN → 0` as a guard.

In [ ]:
y = meta[Y_COLS].fillna(0).astype(np.float32).values
assert set(np.unique(y)).issubset({0.,1.}), np.unique(y)
print('y', y.shape, '| positives/label:')
print(dict(zip(LABELS, y.sum(0).astype(int))))

## C7 · Stream `img_*` → `cxr_emb.npy`

The 768-wide embedding is written to a memmapped `.npy` in chunks so RAM stays
flat regardless of N (PLAN §5.1: `.npy` preferred over 768 parquet columns).

In [ ]:
CXR_PATH = os.path.join(OUT_DIR, 'cxr_emb.npy')
cxr = np.lib.format.open_memmap(CXR_PATH, mode='w+', dtype=np.float32, shape=(N,768))
CHUNK = 20_000; done = 0
for ch in pd.read_csv(CSV_PATH, usecols=IMG_COLS, chunksize=CHUNK,
                      nrows=(QUICK_N if QUICK else None)):
    ch = ch[IMG_COLS]                       # enforce img_0..img_767 order
    cxr[done:done+len(ch)] = ch.values.astype(np.float32)
    done += len(ch); print(f'  {done}/{N}', end='\r')
cxr.flush()
print('\ncxr_emb.npy written', cxr.shape, '| finite:', bool(np.isfinite(cxr[:1000]).all()))

## C8 · Save the new dataset

Row order is identical across `cxr_emb.npy`, `ehr_arrays.npz`, and
`ehr_meta.parquet` — align by position.

In [ ]:
np.savez_compressed(os.path.join(OUT_DIR,'ehr_arrays.npz'),
                    labs=labs, labs_mask=labs_mask, y=y)

meta_out = meta[['dicom_id','subject_id','study_id','split','setting','view']].copy()
meta_out['org_ids']  = org_ids
meta_out['spec_ids'] = spec_ids
meta_out.to_parquet(os.path.join(OUT_DIR,'ehr_meta.parquet'), index=False)

config = dict(
    labs_cols=LABS_COLS, cont_cols=CONT_COLS, flag_cols=flag_cols_out,
    L=L, cxr_dim=768, labels=LABELS, y_cols=Y_COLS,
    reserved=dict(PAD=PAD, UNK=UNK, MISSING=MISSING),
    org_vocab=org_vocab, spec_vocab=spec_vocab,
    org_vocab_size=len(org_vocab['col2id'])+3,
    spec_vocab_size=len(spec_vocab['col2id'])+3,
    cont_stats=cont_stats, log_cols=sorted(LOG_COLS), winsor_pct=WINSOR_PCT)
with open(os.path.join(OUT_DIR,'ehr_config.pkl'),'wb') as f:
    pickle.dump(config, f)

# class_freq.pkl for Distribution-Balanced Loss (PLAN §5.7) — train positives
class_freq = dict(class_freq=y[train_mask].sum(0).astype(int),
                  neg_class_freq=(train_mask.sum()-y[train_mask].sum(0)).astype(int),
                  num_train=int(train_mask.sum()))
with open(os.path.join(OUT_DIR,'class_freq.pkl'),'wb') as f:
    pickle.dump(class_freq, f)

print('saved to', OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    print(' ', fn, round(os.path.getsize(os.path.join(OUT_DIR,fn))/1e6,2),'MB')

## C9 · Sanity checks (PLAN §7)

Shapes, ragged multi-value, vocab rarity ordering, and a reload round-trip.

In [ ]:
cfg = pickle.load(open(os.path.join(OUT_DIR,'ehr_config.pkl'),'rb'))
mp  = pd.read_parquet(os.path.join(OUT_DIR,'ehr_meta.parquet'))
ar  = np.load(os.path.join(OUT_DIR,'ehr_arrays.npz'))
cx  = np.load(os.path.join(OUT_DIR,'cxr_emb.npy'), mmap_mode='r')

assert cx.shape == (len(mp),768)
assert ar['labs'].shape == (len(mp), cfg['L'])
assert ar['y'].shape == (len(mp),14)
assert all(len(o)>=1 for o in mp['org_ids'])                 # never empty
assert org_vocab['col2id'] and min(org_vocab['id2name']) == 0
print('shapes OK · N =', len(mp), '· L =', cfg["L"],
      '· org_vocab =', cfg['org_vocab_size'], '· spec_vocab =', cfg['spec_vocab_size'])
print('example ragged org_ids[:5]:', list(mp['org_ids'][:5]))
print('all checks ✔')

## C10 · (optional) Torch `Dataset` + `EmbeddingBag` collate

The payoff of the redesign: ragged `org_ids`/`spec_ids` batch as flat index +
offsets, exactly what `nn.EmbeddingBag` consumes (PLAN §5.2/§5.3). This is a
consumption demo — the fusion model itself lives in the training notebook.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class EHRFusionDS(Dataset):
    def __init__(self, out_dir, split):
        self.meta = pd.read_parquet(os.path.join(out_dir,'ehr_meta.parquet'))
        ar = np.load(os.path.join(out_dir,'ehr_arrays.npz'))
        self.labs, self.mask, self.y = ar['labs'], ar['labs_mask'], ar['y']
        self.cxr = np.load(os.path.join(out_dir,'cxr_emb.npy'), mmap_mode='r')
        self.idx = np.where(self.meta['split'].values==split)[0]
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        i = self.idx[k]
        return dict(x_cxr=torch.from_numpy(np.asarray(self.cxr[i]).copy()),
                    org=torch.tensor(self.meta['org_ids'].iloc[i], dtype=torch.long),
                    spec=torch.tensor(self.meta['spec_ids'].iloc[i], dtype=torch.long),
                    labs=torch.from_numpy(self.labs[i]),
                    labs_mask=torch.from_numpy(self.mask[i]),
                    y=torch.from_numpy(self.y[i]))

def bag(seqs):                                   # -> flat values + offsets
    offs = torch.tensor([0]+[len(s) for s in seqs][:-1]).cumsum(0)
    return torch.cat(seqs), offs

def collate(batch):
    return dict(
        x_cxr=torch.stack([b['x_cxr'] for b in batch]),
        org=bag([b['org'] for b in batch]),
        spec=bag([b['spec'] for b in batch]),
        labs=torch.stack([b['labs'] for b in batch]),
        labs_mask=torch.stack([b['labs_mask'] for b in batch]),
        y=torch.stack([b['y'] for b in batch]))

ds = EHRFusionDS(OUT_DIR,'train')
dl = DataLoader(ds, batch_size=8, shuffle=True, collate_fn=collate)
b = next(iter(dl))
org_vals, org_offs = b['org']
eb = torch.nn.EmbeddingBag(cfg['org_vocab_size'], 16, mode='sum')
print('batch x_cxr', tuple(b['x_cxr'].shape), '| labs', tuple(b['labs'].shape),
      '| y', tuple(b['y'].shape))
print('EmbeddingBag(org) ->', tuple(eb(org_vals, org_offs).shape))
print('demo OK ✔')